In [1]:
##### Converts raw rasters on cropland and crop production into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # $ of crop production per HA of cropland 
    # tonnes of crop production per HA of cropland

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from rasterio.features import geometry_mask

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# cropland raster
cropland = f"{cd}/Data/Raw/Predictors/Ag_land_ME/crop_land_ha_2020.tif"

# crop production raster
crop_production_USD = f"{cd}/Data/Clean/Production/crop_production_USD_2020.tif"
crop_production_tonnes = f"{cd}/Data/Clean/Production/crop_production_tonnes_2020.tif"

# save paths
pixels_USD = f"{cd}/Data/Clean/Predictors/Rasters/USD_production_per_HA.tif"
pixels_USD_log = f"{cd}/Data/Clean/Predictors/Rasters/log_USD_production_per_HA.tif"

pixels_t = f"{cd}/Data/Clean/Predictors/Rasters/tonnes_production_per_HA.tif"
pixels_t_log = f"{cd}/Data/Clean/Predictors/Rasters/log_tonnes_production_per_HA.tif"

vectors = f"{cd}/Data/Clean/Predictors/Vectors/production_per_HA.csv"

In [5]:
#### Run resampling for pixel scale 

### PREP 
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()
    ref_nodata    = ref.nodata
    ref_data      = ref.read(1)

# build reference valid mask
if ref_nodata is not None:
    ref_valid = ~np.isnan(ref_data) if np.isnan(ref_nodata) else (ref_data != ref_nodata)
else:
    ref_valid = ~np.isnan(ref_data)

# reproject countries once
if countries.crs != dst_crs:
    countries = countries.to_crs(dst_crs)
countries = countries.reset_index(drop=True)

### RESAMPLE 
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            src_nodata=src.nodata,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

land       = resample_to_ref(cropland)
production_USD = resample_to_ref(crop_production_USD)
production_tonnes = resample_to_ref(crop_production_tonnes)

In [6]:
### CALCULATE USD INTENSITY
with np.errstate(invalid="ignore", divide="ignore"):
    intensity = np.where(
        (land > 0) & ~np.isnan(land) & ~np.isnan(production_USD),
        production_USD / land,
        np.nan
    ).astype(np.float32)

# clip at 99th percentile
p99 = np.nanpercentile(intensity, 99)
intensity = np.clip(intensity, 0, p99).astype(np.float32)

### ALIGN TO REFERENCE MASK AND GAP-FILL

# Step 1: blank pixels outside reference mask
intensity[~ref_valid] = np.nan

# Step 2: identify pixels needing fill
needs_fill = ref_valid & np.isnan(intensity)
print(f"Pixels needing fill: {needs_fill.sum()}")

if needs_fill.any():

    country_stats = rasterstats.zonal_stats(
        countries,
        intensity,
        affine=dst_transform,
        stats=["mean"],
        nodata=np.nan,
    )

    country_means = {
        i: s["mean"] for i, s in enumerate(country_stats)
        if s["mean"] is not None
    }

    fill_array = np.full(dst_shape, np.nan, dtype=np.float32)

    for idx, row in countries.iterrows():
        mean_val = country_means.get(idx)
        if mean_val is None:
            continue

        country_mask = geometry_mask(
            [row.geometry],
            transform=dst_transform,
            invert=True,
            out_shape=dst_shape,
        )

        fill_array[needs_fill & country_mask] = mean_val

    intensity = np.where(needs_fill, fill_array, intensity).astype(np.float32)

    still_missing = ref_valid & np.isnan(intensity)
    if still_missing.any():
        print(f"Warning: {still_missing.sum()} pixels still unfilled "
              f"(country had no valid intensity data). These will be saved as NoData.")

### SAVE 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = intensity.copy()
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_USD, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

print(f"Saved → {pixels_USD}")

Pixels needing fill: 1035476
Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/USD_production_per_HA.tif


In [7]:
### CALCULATE TONNE INTENSITY
with np.errstate(invalid="ignore", divide="ignore"):
    intensity = np.where(
        (land > 0) & ~np.isnan(land) & ~np.isnan(production_tonnes),
        production_tonnes / land,
        np.nan
    ).astype(np.float32)

# clip at 99th percentile
p99 = np.nanpercentile(intensity, 99)
intensity = np.clip(intensity, 0, p99).astype(np.float32)

### ALIGN TO REFERENCE MASK AND GAP-FILL

# Step 1: blank pixels outside reference mask
intensity[~ref_valid] = np.nan

# Step 2: identify pixels needing fill
needs_fill = ref_valid & np.isnan(intensity)
print(f"Pixels needing fill: {needs_fill.sum()}")

if needs_fill.any():

    country_stats = rasterstats.zonal_stats(
        countries,
        intensity,
        affine=dst_transform,
        stats=["mean"],
        nodata=np.nan,
    )

    country_means = {
        i: s["mean"] for i, s in enumerate(country_stats)
        if s["mean"] is not None
    }

    fill_array = np.full(dst_shape, np.nan, dtype=np.float32)

    for idx, row in countries.iterrows():
        mean_val = country_means.get(idx)
        if mean_val is None:
            continue

        country_mask = geometry_mask(
            [row.geometry],
            transform=dst_transform,
            invert=True,
            out_shape=dst_shape,
        )

        fill_array[needs_fill & country_mask] = mean_val

    intensity = np.where(needs_fill, fill_array, intensity).astype(np.float32)

    still_missing = ref_valid & np.isnan(intensity)
    if still_missing.any():
        print(f"Warning: {still_missing.sum()} pixels still unfilled "
              f"(country had no valid intensity data). These will be saved as NoData.")

### SAVE 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = intensity.copy()
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_t, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

print(f"Saved → {pixels_t}")

Pixels needing fill: 1089888
Saved → /Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Predictors/Rasters/tonnes_production_per_HA.tif


In [8]:
### Calculate log intensities 

def make_log_raster(in_path, out_path):
    with rasterio.open(in_path) as src:
        arr = src.read(1)
        meta = src.meta.copy()
        nodata = src.nodata

    # mask nodata + non-positive values (log undefined for <= 0)
    if nodata is not None and not np.isnan(nodata):
        valid = (arr != nodata) & (arr > 0)
    else:
        valid = (~np.isnan(arr)) & (arr > 0)

    with np.errstate(divide="ignore", invalid="ignore"):
        log_arr = np.where(valid, np.log(arr), np.nan).astype(np.float32)

    meta.update(dtype=rasterio.float32, count=1, nodata=np.nan)

    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(log_arr, 1)

make_log_raster(pixels_USD, pixels_USD_log)
make_log_raster(pixels_t, pixels_t_log)


In [10]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_production_USD  = rasterstats.zonal_stats(gdf_proj, crop_production_USD, stats="sum", nodata=-9999)
zonal_production_tonnes  = rasterstats.zonal_stats(gdf_proj, crop_production_tonnes, stats="sum", nodata=-9999)
zonal_land = rasterstats.zonal_stats(gdf_proj, cropland,   stats="sum", nodata=-9999)

### compute share at vector level
production_USD_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_production_USD])
production_tonne_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_production_tonnes])
land_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_land])

with np.errstate(invalid="ignore", divide="ignore"):
    int_USD = np.where(
        (land_sums > 0) & ~np.isnan(land_sums) & ~np.isnan(production_USD_sums),
        production_USD_sums / land_sums,
        np.nan
    )

result["USD_production_per_HA"] = int_USD
p99 = result["USD_production_per_HA"].quantile(0.99)
result["USD_production_per_HA"] = result["USD_production_per_HA"].clip(0, p99)

with np.errstate(invalid="ignore", divide="ignore"):
    int_tonnes = np.where(
        (land_sums > 0) & ~np.isnan(land_sums) & ~np.isnan(production_tonne_sums),
        production_tonne_sums / land_sums,
        np.nan
    )

result["tonnes_production_per_HA"] = int_tonnes
p99 = result["tonnes_production_per_HA"].quantile(0.99)
result["tonnes_production_per_HA"] = result["tonnes_production_per_HA"].clip(0, p99)

### save
result.to_csv(vectors, index=False)